# 🧠 Notebook 3: LangChain Memory
### Phase 3 – Advanced LangChain Concepts

---

## Why Memory Matters

By default, LLMs are **stateless** — every call is independent.  
Without memory:
```
User: My name is Bakar
AI:   Nice to meet you, Bakar!
User: What is my name?
AI:   I don't know your name. (❌ Forgot!)
```
With memory:
```
User: My name is Bakar
AI:   Nice to meet you, Bakar!
User: What is my name?
AI:   Your name is Bakar. (✅ Remembered!)
```

## Three Types of Memory

| Type | How it works | Best for |
|---|---|---|
| **ConversationBufferMemory** | Stores ALL messages verbatim | Short sessions |
| **ConversationEntityMemory** | Extracts & tracks named entities | Personal assistants |
| **VectorStoreRetrieverMemory** | Semantic search over past messages | Long-term memory |


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
print('API Key:', '✅' if GROQ_API_KEY and GROQ_API_KEY != 'your_groq_api_key_here' else '❌')

---
## 1. ConversationBufferMemory

The simplest memory. Stores EVERY message in a rolling buffer and injects
the full history into every LLM call.

**Trade-off**: Works perfectly but gets expensive (more tokens) as chats grow.

In [ ]:
from langchain.memory import ConversationBufferMemory

# Create memory
memory = ConversationBufferMemory(
    memory_key='chat_history',
    return_messages=True,  # Returns Message objects (HumanMessage, AIMessage)
)

# Simulate 3 turns of conversation
memory.save_context({'input': 'My name is Bakar'}, {'output': 'Great name, Bakar!'})
memory.save_context({'input': 'I am learning LangChain'}, {'output': 'LangChain is powerful!'})
memory.save_context({'input': 'I like AI'}, {'output': 'AI is fascinating!'})

# Load and inspect
history = memory.load_memory_variables({})
print(f'Stored {len(history["chat_history"])} messages:\n')
for msg in history['chat_history']:
    print(f'  [{msg.type.upper()}] {msg.content}')

In [ ]:
# ── Use BufferMemory in a real chat chain ───────────────────────────────────
from langchain.chains import ConversationChain
from langchain_groq import ChatGroq

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.5, groq_api_key=GROQ_API_KEY)

chat_memory = ConversationBufferMemory()
chain = ConversationChain(llm=llm, memory=chat_memory, verbose=False)

# Multi-turn chat
resp1 = chain.predict(input='My name is Bakar and I love Python.')
print('Turn 1:', resp1[:120])

resp2 = chain.predict(input='What language did I say I love?')
print('Turn 2:', resp2[:120])  # Should mention Python!

---
## 2. ConversationEntityMemory

Uses an LLM to **extract and track named entities** from the conversation.
Instead of storing raw text, it builds a structured entity store:

```json
{
  "Elon Musk": "Founder of Tesla, SpaceX. Born in South Africa.",
  "SpaceX":    "Rocket company founded in 2002 by Elon Musk."
}
```

Great for: CRM bots, personal assistants, character tracking.

In [ ]:
from langchain.memory import ConversationEntityMemory

entity_memory = ConversationEntityMemory(
    llm=llm,
    return_messages=True
)

# Add conversation turns
entity_memory.save_context(
    {'input': 'Elon Musk founded SpaceX in 2002 to reduce space travel costs.'},
    {'output': 'Yes, SpaceX is revolutionising space exploration.'}
)
entity_memory.save_context(
    {'input': 'SpaceX is headquartered in Hawthorne, California.'},
    {'output': 'That is correct!'}
)

# Get entity store
variables = entity_memory.load_memory_variables({'input': ''})
print('Entity Store:')
for entity, info in variables.get('entities', {}).items():
    print(f'  [{entity}]: {info}')

---
## 3. VectorStoreRetrieverMemory (Semantic Long-Term Memory)

Embeds past messages as **vectors** and retrieves the most **semantically similar** ones at query time.

**Key advantage**: Doesn't matter how many past messages there are —  
only the most relevant ones are retrieved. Scales to thousands of turns.

```
Past messages stored as embeddings in ChromaDB
          ┌─────────────────────────────────────┐
          │   [v1] "Tell me about LangChain"     │
          │   [v2] "What is a vector database?"  │
          │   [v3] "Explain Python decorators"   │
          └─────────────────────────────────────┘
                            │
Query: "AI frameworks" ──►  Similarity Search
                            │
              Returns: [v1, v2] (most similar)
```

In [ ]:
from langchain.memory import VectorStoreRetrieverMemory
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Free local embeddings (no API key needed)
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

# Create in-memory ChromaDB vector store (no persist directory = in-memory only)
vectorstore = Chroma(
    embedding_function=embeddings,
    collection_name='memory_demo',
)

# Build memory with semantic retriever
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})
vector_memory = VectorStoreRetrieverMemory(retriever=retriever)

# Store past conversations
facts = [
    ('What is LangChain?', 'LangChain is a framework for building LLM-powered apps.'),
    ('What is ChromaDB?', 'ChromaDB is an open-source vector database for embeddings.'),
    ('What is a transformer?', 'Transformers are neural networks using self-attention mechanisms.'),
    ('Who is Bakar?', 'Bakar is a developer learning LangChain.')
]

for human, ai in facts:
    vector_memory.save_context({'input': human}, {'output': ai})

# Semantic retrieval — finds similar past messages
query = 'Tell me about AI frameworks and libraries'
result = vector_memory.load_memory_variables({'prompt': query})
print(f'Query: "{query}"')
print('\nSemantically similar past conversation:')
print(result.get('history', 'No match found.'))

---
## Memory Type Comparison

| Feature | Buffer | Entity | Vector |
|---|---|---|---|
| LLM call required? | ❌ No | ✅ Yes | ❌ No |
| Scales to long chats? | ❌ No | ⚠️ Partial | ✅ Yes |
| Structured output? | ❌ No | ✅ Yes | ❌ No |
| Semantic search? | ❌ No | ❌ No | ✅ Yes |
| Best for | Demos, short chats | Entity tracking | Long-term memory |

## How Memory Integrates with Agents

```python
# The app.py uses this pattern:
memory = ConversationBufferMemory()

# Agent processes the question
answer = agent.invoke({"input": question})

# Save to memory AFTER the agent responds
memory.save_context({"input": question}, {"output": answer})

# On next turn: inject history into agent context
history = memory.load_memory_variables({})["chat_history"]
context_question = f"{history}\n\nCurrent: {new_question}"
answer = agent.invoke({"input": context_question})
```

**👉 Next:** Run `python app.py` to see all three concepts (Agents + Tools + Memory) working together!
